# 3.7 — VPC Direct: Binary Classification with the VPC Model

This notebook demonstrates binary classification using the high-level `VPC` class from `PhasorFlow.models`. Unlike notebooks 3.1–3.4 which build the circuit from scratch, here we use the **scikit-learn-style API** (`fit`, `predict`, `score`) that wraps all circuit construction internally.

We will:
1. Generate a synthetic phase-based dataset
2. Split into train/test sets
3. Instantiate a `VPC` model
4. Train with `model.fit()`
5. Evaluate with `model.predict()` and `model.score()`
6. Visualize results

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import math
import numpy as np
import torch
import matplotlib.pyplot as plt

import PhasorFlow as pf
from PhasorFlow.models import VPC

np.random.seed(42)
torch.manual_seed(42)

## 1. Synthetic Dataset

We generate 200 samples with N=12 features as random phases in $[0, 2\pi]$. The binary label is determined by a non-linear cosine-sum boundary:

$$y = \begin{cases} 1 & \text{if } \sum_{i=1}^{N} \cos(x_i) > 0 \\ 0 & \text{otherwise} \end{cases}$$

In [ ]:
N_FEATURES = 12
N_SAMPLES = 200

# Generate random phases in [0, 2π]
X = torch.empty(N_SAMPLES, N_FEATURES).uniform_(0, 2 * math.pi)

# Non-linear cosine-sum boundary
true_signals = torch.sum(torch.cos(X), dim=1)
y = (true_signals > 0).float()

print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Class distribution: {int(y.sum())} positive, {int(len(y) - y.sum())} negative')

In [ ]:
plt.figure(figsize=(10, 4))
plt.scatter(X[:, 0].numpy(), X[:, 1].numpy(), c=y.numpy(), cmap='coolwarm', edgecolors='k', linewidth=0.5)
plt.xlabel('Feature 1 (Phase)')
plt.ylabel('Feature 2 (Phase)')
plt.title('Synthetic Phase Data — Non-linear Cosine-Sum Boundary')
plt.colorbar(label='Class Label')
plt.tight_layout()
plt.show()

## 2. Train / Test Split

In [ ]:
split = int(0.8 * N_SAMPLES)  # 80/20 split
indices = torch.randperm(N_SAMPLES)

X_train, y_train = X[indices[:split]], y[indices[:split]]
X_test, y_test = X[indices[split:]], y[indices[split:]]

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')

## 3. Create & Inspect VPC Model

The `VPC` class encapsulates the entire phasor circuit construction. Key parameters:

| Parameter | Description |
|---|---|
| `num_features` | Number of input features (= oscillator threads) |
| `num_layers` | Number of variational layers per stack |
| `num_classes` | 2 for binary (default) |
| `coupling` | `'mix_dft'`, `'mix_only'`, or `'dft_only'` |
| `num_stacks` | Number of stacked VPC blocks |
| `inter_stack` | Operation between stacks: `'none'`, `'pullback'`, `'threshold'` |

In [ ]:
# Single-stack VPC for binary classification
model = VPC(
    num_features=N_FEATURES,
    num_layers=2,
    num_classes=2,
    coupling='mix_dft',
)
print(model)
print(f'\nTotal trainable parameters: {model.total_params}')

### Circuit Visualization

We can inspect the underlying phasor circuit that the VPC constructs:

In [ ]:
# Get the circuit for a sample input
sample_circuit = model.get_circuit(X_train[0])
print(f'Circuit: {sample_circuit.num_threads} threads, {len(sample_circuit.operations)} operations')
pf.draw(sample_circuit, mode='text')

## 4. Training

The `fit()` method handles the entire training loop: forward pass → loss → backward → optimizer step.

In [ ]:
result = model.fit(
    X_train, y_train,
    epochs=50,
    lr=0.1,
    optimizer_type='adam',
    verbose=True,
    print_every=5,
)
print(f'\nFinal loss: {result["final_loss"]:.4f}')

### Training Loss Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(result['loss_history'], color='tab:blue', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('VPC Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluation

The `predict()` and `score()` methods provide scikit-learn-compatible interfaces.

In [ ]:
# Predictions
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)

# Accuracy
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print(f'Train accuracy: {train_acc:.2%}')
print(f'Test accuracy:  {test_acc:.2%}')

## 6. Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train set
ax = axes[0]
correct_train = (train_preds == y_train)
ax.scatter(
    X_train[correct_train, 0].numpy(), X_train[correct_train, 1].numpy(),
    c=y_train[correct_train].numpy(), cmap='coolwarm', marker='o', edgecolors='k',
    linewidth=0.5, label='Correct'
)
ax.scatter(
    X_train[~correct_train, 0].numpy(), X_train[~correct_train, 1].numpy(),
    c='gray', marker='x', s=80, linewidths=2, label='Incorrect'
)
ax.set_title(f'Train (Accuracy: {train_acc:.1%})')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend()

# Test set
ax = axes[1]
correct_test = (test_preds == y_test)
ax.scatter(
    X_test[correct_test, 0].numpy(), X_test[correct_test, 1].numpy(),
    c=y_test[correct_test].numpy(), cmap='coolwarm', marker='o', edgecolors='k',
    linewidth=0.5, label='Correct'
)
ax.scatter(
    X_test[~correct_test, 0].numpy(), X_test[~correct_test, 1].numpy(),
    c='gray', marker='x', s=80, linewidths=2, label='Incorrect'
)
ax.set_title(f'Test (Accuracy: {test_acc:.1%})')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend()

plt.tight_layout()
plt.show()

## 7. Summary

This notebook demonstrated binary classification using the high-level `VPC` model from `PhasorFlow.models`:

```python
from PhasorFlow.models import VPC

model = VPC(num_features=12, num_layers=2)
model.fit(X_train, y_train, epochs=50, lr=0.1)
accuracy = model.score(X_test, y_test)
```

The `VPC` class handles all circuit construction internally:
- Data encoding into oscillator phases
- Trainable shift layers with Mix + DFT coupling
- Sine/cosine envelope readout for binary decision
- MSE loss and gradient-based optimization

For first-principles implementations, see notebooks **3.1–3.4**.

In [ ]:
print('=== VPC Binary Classification Summary ===')
print(f'Features:      {N_FEATURES}')
print(f'Layers:        2')
print(f'Coupling:      mix_dft')
print(f'Params:        {model.total_params}')
print(f'Train samples: {X_train.shape[0]}')
print(f'Test samples:  {X_test.shape[0]}')
print(f'Train acc:     {train_acc:.2%}')
print(f'Test acc:      {test_acc:.2%}')